# AgentCore Observability — Monitoring Your Composable System

Now that you have deployed the full multi-agent system (Order Agent, Refund Agent, Orchestrator Harness,
Gateway, and Memory), it's time to observe how it behaves in production.

---

In this notebook you will learn how to:
- Navigate the **GenAI Observability dashboard** in Amazon CloudWatch
- Monitor **Agent metrics** (sessions, invocations, tokens, errors)
- Monitor **Gateway metrics** (invocations, latency, throttles)
- Monitor **Memory metrics** (events, extracted records)
- View **Sessions** to understand user interaction patterns
- Drill into **Traces** to see the full execution path of a request
- Use **Transaction Search** to query spans across all resources
- Access **Application Logs** for detailed debugging

---

## Prerequisites

Before exploring observability, make sure you have:
1. Completed all L3 notebooks (pre-requisites, order agent, refund agent, orchestrator)
2. Invoked the chatbot at least 2-3 times to generate telemetry data
3. Waited 2-3 minutes after invocations for data to propagate to CloudWatch

---

## What is AgentCore Observability?

Amazon Bedrock AgentCore Observability provides **built-in monitoring** for your AI agents with
minimal setup. It gives you near real-time visibility into agent performance through metrics, traces,
and logs — all powered by Amazon CloudWatch.

### Why It's Easy

- **Harness (Orchestrator)**: Observability is **fully automatic**. Every invocation generates traces,
  logs, and metrics with zero configuration. Model calls, tool invocations, memory operations — each
  step appears with timing and payload details from the first invocation.

- **Gateway, Memory, Runtime agents**: Enable observability by configuring **log deliveries** (APPLICATION_LOGS,
  USAGE_LOGS) and **tracing** through the CloudWatch Vended Logs Delivery API. Once enabled, telemetry
  flows automatically.

- **Transaction Search**: A one-time account-level setup that routes all trace spans to CloudWatch.
  Once enabled, every AgentCore resource that emits spans becomes searchable and visualizable in the
  GenAI Observability dashboard.

### What Gets Captured Automatically

| Component | Metrics | Traces | Logs |
|-----------|---------|--------|------|
| **Harness** | Sessions, invocations, tokens, latency | Full execution path (model → tool → memory) | Automatic |
| **Gateway** | Invocations, errors, throttles, latency | Tool call spans | Via log delivery |
| **Memory** | API calls, extracted records, latency | Read/write spans | Via log delivery |
| **Runtime Agents** | Sessions, invocations, vCPU, memory | Agent execution spans | Via log delivery |


### How We Enabled It in This Workshop

In the L3 pre-requisites notebook (Step 2), we enabled **Transaction Search** — this is the foundation
that allows all spans to flow into CloudWatch. Then for each resource (Gateway, agents, Memory), we
configured log deliveries and tracing using the CloudWatch Vended Logs Delivery API.

The Harness required no additional setup — AgentCore handles its observability internally.

---

## How to Access the Dashboard

1. Open the **AWS Console** → **CloudWatch**
2. In the left navigation, expand **GenAI Observability**
3. Click **Bedrock AgentCore**

You will see tabs for: **Agents**, **Built-in tools**, **Gateways**, **Identity**, **Memory**, **Payments**


## License Details

Refer to LICENSE file in the main folder for license details.

---
## 1. Agents — Runtime Performance

The **Agents** tab shows metrics for all agents deployed on AgentCore Runtime. This is your
top-level view of agent health.

**What to look for:**
- **Agents/Endpoints**: Number of deployed agents with observability enabled
- **Sessions**: Total user sessions across all agents
- **Traces**: Number of execution traces captured
- **Total tokens**: Aggregate token consumption across all model calls
- **Error rate**: Percentage of failed invocations

**Navigation:** CloudWatch → GenAI Observability → Bedrock AgentCore → **Agents** tab

![Agents Overview](agent_metrics.png)


## 2. Harness (Orchestrator) — Agent Metrics

The Harness is the orchestrator that coordinates all specialist agents. Its observability is fully automatic — no configuration needed.

**Overview metrics** — sessions, invocations, total tokens consumed, and error rate across all Harness calls.

![Harness Overview](harness_metrics1.png)

**Sessions & Traces** — each conversation creates a session; each turn within it creates a trace. Click a session to see the full conversation flow.

![Sessions and Traces](harness_metrics2.png)

**Inbound Authentication** — shows how requests are authenticated (SigV4, OAuth) before reaching the Harness.

![Inbound Auth](harness_metrics3.png)

**Runtime Metrics** — Runtime metrics of the Harness compute environment.

![Runtime Metrics](harness_metrics4.png)

**Trace Timeline** — the full execution path of a single Harness invocation: model call → tool selection → agent delegation → memory write. Each span shows timing and status.

![Harness Trace](harness_metrics5.png)

**Request & Response Payloads** — expand any span to see the exact input sent to the model and the output returned. Useful for debugging unexpected agent behavior.

![Trace Payloads](harness_metrics6.png)

---
## 2. Gateways  — Tool Routing Performance

The **Gateways** tab monitors the AgentCore Gateway that routes tool calls from agents to
Lambda functions. Every time an agent calls `check_order_details` or `refund_policy`, it goes
through the Gateway.

**What to look for:**
- **Invocations**: Total tool calls routed through the Gateway
- **Error rate**: Failed tool calls (Lambda errors, auth failures, timeouts)
- **Throttle rate**: Requests rejected due to rate limiting
- **Avg. latency**: Time from Gateway receiving the request to returning the response

**Navigation:** CloudWatch → GenAI Observability → Bedrock AgentCore → **Gateways** tab

![Gateway Metrics](gateway_metrics.png)


---
## 3. Memory — Conversation State

The **Memory** tab shows how AgentCore Memory is being used to maintain conversation context.
The Harness reads from memory at the start of each turn and writes extracted knowledge after.

**What to look for:**
- **Create events (short-term memory)**: API calls to store chat messages — one per user/assistant turn
- **Extracted memory records**: Long-term facts extracted by the memory strategies (preferences, summaries)
- **Latency**: Time for memory read/write operations
- **Error rate**: Failed memory operations

**Navigation:** CloudWatch → GenAI Observability → Bedrock AgentCore → **Memory** tab

![Memory Metrics](memory_metrics.png)

---
## 4. All Sessions — User Interaction Patterns

The **All Sessions** view groups traces by session ID, giving you a conversation-level view.
Each session represents one user's interaction with the chatbot.

**What to look for:**
- **Session duration**: How long users interact before leaving
- **Traces per session**: Number of turns in a conversation
- **Error sessions**: Sessions that encountered failures

**Navigation:** CloudWatch → GenAI Observability → Bedrock AgentCore → **All sessions** tab

![All Sessions](allsessions.png)



---
## 5. All Traces — End-to-End Request Flow

The **All Traces** view shows individual execution traces. Each trace represents one complete
request-response cycle through the system.

**Navigation:** CloudWatch → GenAI Observability → Bedrock AgentCore → **All traces** tab

![All Traces](alltraces.png)

**Drilling into a trace:**

Click on any trace ID to see the full span timeline. This shows every step the system took:
- Model invocations (with input/output tokens)
- Tool selections and calls
- Memory reads and writes
- Agent-to-agent delegations

![Trace Detail](trace_response.png)


---
## 6. Model Invocations — LLM Usage

The **Model Invocations** view (under GenAI Observability) shows all Bedrock model calls
made by your agents. This is useful for understanding token consumption and model performance.

**What to look for:**
- **Input/Output tokens per call**: How much context is being sent and generated
- **Latency per model call**: Time to first token and total generation time
- **Model ID**: Which model is being used (Claude Sonnet, Haiku, etc.)

**Navigation:** CloudWatch → GenAI Observability → **Model Invocations**

![Model Invocations](modelinvocations.png)


---
## 7. Transaction Search — Query Spans Across Resources

**Transaction Search** lets you search and filter spans across all AgentCore resources.
This is the most powerful debugging tool — you can find specific requests by trace ID,
filter by error status, or search by time range.

**Navigation:** CloudWatch → Application Signals → **Transaction Search**

![Transaction Search](transaction_search.png)



---
## 8. Spans, Tool Logs & Application Logs — Detailed Debugging

When high-level metrics aren't enough, dive deeper into **spans**, **tool execution logs**,
and **application logs** for full visibility into what happened during a request.

### 8.1 Spans (from Transaction Search)

Spans are the building blocks of traces. Each span represents a single operation — a model call,
a tool invocation, a memory read, or a Gateway routing decision. Transaction Search stores all
spans in the `aws/spans` log group.

**How to query spans:**
1. CloudWatch → Log groups → `aws/spans`
2. Click **Search log group** or use **Logs Insights**
3. Filter by trace ID, service name, or error status

**Example Logs Insights query:**
```
fields @timestamp, traceId, name, duration, statusCode
| filter name like /gateway/ or name like /tool/
| sort @timestamp desc
| limit 50
```

Each span contains:
- **traceId**: Links all spans in a single request together
- **name**: The operation (e.g., `tools/call`, `bedrock.invoke_model`)
- **duration**: How long the operation took
- **attributes**: Request/response details, tool names, model IDs

### 8.2 Tool Execution Logs (Gateway APPLICATION_LOGS)

The Gateway APPLICATION_LOGS capture every tool call with full request and response payloads.
This is the best place to debug tool failures.

**Log group:** `/aws/vendedlogs/bedrock-agentcore/gateway/APPLICATION_LOGS/{gateway-id}`

Each log entry includes:
- `resource_arn`: The gateway that processed the request
- `event_timestamp`: When the tool was called
- `body.log`: What happened — "Started processing request", "Executing tool order-tools___check_order_details"
- `body.requestBody`: The full MCP request (tool name + arguments)
- `body.responseBody`: The full tool response (data from Lambda/DynamoDB)
- `body.isError`: Whether the tool call failed

**Example:** When a user asks "What's the status of ORD-10001?", you'll see:
1. `Started processing request` — Gateway receives the MCP call
2. `Executing tool order-tools___check_order_details` — Gateway routes to Lambda
3. Response with the order data from DynamoDB

### 8.3 Agent Runtime Logs

Runtime APPLICATION_LOGS show agent startup, configuration loading, and invocation details.

**Log group:** `/aws/vendedlogs/bedrock-agentcore/runtime/APPLICATION_LOGS/{runtime-id}`

Useful for debugging:
- Agent startup failures (missing SSM parameters, import errors)
- Cognito token acquisition
- Gateway connection issues

### 8.4 Where to Find Logs

| Resource | Log Group | What It Shows |
|----------|----------|---------------|
| Gateway (tools) | `/aws/vendedlogs/.../gateway/APPLICATION_LOGS/{gateway-id}` | Tool call requests/responses, routing decisions |
| Order Agent | `/aws/vendedlogs/.../runtime/APPLICATION_LOGS/{runtime-id}` | Agent startup, config, invocations |
| Refund Agent | `/aws/vendedlogs/.../runtime/APPLICATION_LOGS/{runtime-id}` | Agent startup, config, invocations |
| Memory | `/aws/vendedlogs/.../memory/APPLICATION_LOGS/{memory-id}` | Memory read/write operations |
| Spans (all) | `aws/spans` | OpenTelemetry spans from all resources |

**Navigation:** CloudWatch → Log groups → search for `vendedlogs/bedrock-agentcore`

![Log Groups](loggroups.png)

![Log Events](logevents.png)



---
## Summary

| What to Monitor | Where to Look | Key Metric |
|----------------|---------------|------------|
| Agent health | Agents tab | Error rate, sessions |
| Tool performance | Gateways tab | Latency, invocations |
| Conversation state | Memory tab | Events, extracted records |
| User patterns | All Sessions | Session count, duration |
| Request debugging | All Traces → Trace detail | Span timeline, payloads |
| Token costs | Model Invocations | Input/output tokens |
| Cross-resource search | Transaction Search | Filtered spans |
| Detailed errors | Application Logs | Stack traces, payloads |
